# 📈 Polynomial Regression from Scratch

**Dataset:** California Housing — predicting median house values  
**Method:** Degree-2 polynomial regression via two approaches:
1. **Gradient Descent** — iterative weight updates using MSE gradients  
2. **Normal Equation** — closed-form optimal solution via pseudoinverse  

## The Model

$$\hat{y} = X^2 \cdot W_2 + X \cdot W_1 + b$$

## The Loss (MSE)

$$\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2$$

## Gradients

$$\frac{\partial \mathcal{L}}{\partial W_1} = \frac{2}{N} X^T (\hat{y} - y)$$

$$\frac{\partial \mathcal{L}}{\partial W_2} = \frac{2}{N} (X^2)^T (\hat{y} - y)$$

$$\frac{\partial \mathcal{L}}{\partial b} = \frac{2}{N} \sum (\hat{y} - y)$$

## Normal Equation

$$W^* = (X^T X)^{+} X^T y$$

Where $(X^T X)^{+}$ is the Moore-Penrose pseudoinverse —
handles cases where $X^T X$ is singular or near-singular.

In [ ]:
# ── Imports & Config ──────────────────────────────────────────────────────────
import sys, os
sys.path.append(os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import yaml
from sklearn.datasets import fetch_california_housing
from src.polynomial_regression import PolynomialRegression

with open("../configs/regression_config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("Config loaded:")
for section, values in config.items():
    print(f"  {section}: {values}")

In [ ]:
# ── Load & Prepare Data ───────────────────────────────────────────────────────
housing = fetch_california_housing()
X       = housing.data                      # (20640, 8)
y       = housing.target.reshape(-1, 1)    # (20640, 1)

# Z-score standardize X and y
# Why: gradient descent converges faster when all features
# are on the same scale. Without this, features like
# population (thousands) dominate features like rooms (single digits).
X_mean, X_std = np.mean(X, axis=0), np.std(X, axis=0)
y_mean, y_std = np.mean(y, axis=0), np.std(y, axis=0)
X = (X - X_mean) / X_std
y = (y - y_mean) / y_std

# Shuffle with fixed seed for reproducibility
seed = config["reproducibility"]["random_seed"]
rng  = np.random.default_rng(seed)
idx  = rng.permutation(len(X))
X, y = X[idx], y[idx]

# Train / test split
split   = int((1 - config["data"]["test_size"]) * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"X_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"X_test  : {X_test.shape}    y_test  : {y_test.shape}")

In [ ]:
# ── Train Model ───────────────────────────────────────────────────────────────
model = PolynomialRegression(config)
model.fit(X_train, y_train)

print(f"\nFinal training loss : {model.loss_history[-1]:.6f}")
print(f"W1 shape            : {model.W1.shape}")
print(f"W2 shape            : {model.W2.shape}")
print(f"W_star shape        : {model.W_star.shape}")

In [ ]:
# ── Loss Curve ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(model.loss_history, color="#1A6BC4", linewidth=1.8, label="GD Training Loss")
ax.set_xlabel("Epoch",    fontsize=12)
ax.set_ylabel("MSE Loss", fontsize=12)
ax.set_title("Polynomial Regression — Gradient Descent Loss Curve", fontsize=14)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
os.makedirs("../assets", exist_ok=True)
plt.savefig("../assets/loss_curve_regression.png", dpi=150)
plt.show()
print("Saved → assets/loss_curve_regression.png")

In [ ]:
# ── Evaluate: Gradient Descent vs Normal Equation ────────────────────────────
metrics_gd = model.evaluate(X_test, y_test, method="gd")
metrics_ne = model.evaluate(X_test, y_test, method="normal_equation")

print("─" * 48)
print(f"  {'Metric':<22} {'GD':>8}  {'Normal Eq':>10}")
print("─" * 48)
for key in metrics_gd:
    print(f"  {key:<22} {metrics_gd[key]:>8}  {metrics_ne[key]:>10}")
print("─" * 48)

In [ ]:
# ── Predicted vs Actual Plot ──────────────────────────────────────────────────
y_pred_gd = model.predict(X_test, method="gd")
y_pred_ne = model.predict(X_test, method="normal_equation")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, preds, title, color in zip(
    axes,
    [y_pred_gd, y_pred_ne],
    ["Gradient Descent", "Normal Equation"],
    ["#1A6BC4", "#00C28B"]
):
    ax.scatter(y_test, preds, alpha=0.3, s=8, color=color)
    ax.plot([y_test.min(), y_test.max()],
            [y_test.min(), y_test.max()],
            "r--", linewidth=1.5, label="Perfect Prediction")
    ax.set_xlabel("Actual (standardized)",    fontsize=11)
    ax.set_ylabel("Predicted (standardized)", fontsize=11)
    ax.set_title(f"{title} — Predicted vs Actual", fontsize=12)
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../assets/predictions_regression.png", dpi=150)
plt.show()
print("Saved → assets/predictions_regression.png")

## Results Summary

| Method | Test MSE | Test MAE | Test R² |
|---|---|---|---|
| Gradient Descent | *see output above* | *see output above* | *see output above* |
| Normal Equation  | *see output above* | *see output above* | *see output above* |

**Key Takeaways:**
- Normal Equation gives the **exact** optimal solution in one shot  
- Gradient Descent **approximates** it through iterative updates  
- For large datasets, GD scales better — Normal Equation requires  
  inverting a $(2F+1) \times (2F+1)$ matrix which is $O(n^3)$  
- The scatter plot diagonal = perfect prediction line.  
  Tighter clustering around the diagonal = better model.